# Week 6 — Integrated Data Science Project

Problem statement: Can chemical measurements distinguish three wine classes in the Wine Recognition benchmark? This notebook integrates data validation, EDA, statistical testing, model tuning, evaluation and visual communication.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
wine = load_wine(as_frame=True)
df = wine.frame.copy()
df["wine_class"] = df["target"].map({0:"class_0", 1:"class_1", 2:"class_2"})
df = df.drop(columns=["target"])

print("Shape:",df.shape)
print("Missing cells:",df.isna().sum().sum())
print("Duplicate rows:",df.duplicated().sum())
display(df.head())


## Exploratory analysis

In [ ]:
display(df.groupby("wine_class").size().rename("sample_count"))
df.drop(columns="wine_class").hist(figsize=(14,10),bins=15); plt.tight_layout(); plt.show()
plt.figure(figsize=(10,7)); sns.heatmap(df.drop(columns="wine_class").corr(),cmap="coolwarm",center=0); plt.title("Feature Correlation Heatmap"); plt.tight_layout(); plt.show()

## Statistical test — ANOVA on alcohol

In [ ]:
from scipy.stats import f_oneway
groups=[df.loc[df.wine_class==c,"alcohol"] for c in sorted(df.wine_class.unique())]
f_stat,p_value=f_oneway(*groups)
print("F-statistic:",f_stat,"P-value:",p_value)
print("Reject H0 at alpha=0.05" if p_value<0.05 else "Fail to reject H0")

## Train/test split and baseline model comparison

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report,ConfusionMatrixDisplay
X=df.drop(columns="wine_class"); y=df["wine_class"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
models={"Logistic Regression":Pipeline([("scale",StandardScaler()),("model",LogisticRegression(max_iter=3000))]),
        "Random Forest":RandomForestClassifier(random_state=42,class_weight="balanced")}
for name,est in models.items():
    est.fit(X_train,y_train); pr=est.predict(X_test)
    print(name,"accuracy",accuracy_score(y_test,pr),"macro F1",f1_score(y_test,pr,average="macro"))

## Hyperparameter tuning — Random Forest

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
pipe=Pipeline([("model",RandomForestClassifier(random_state=42,class_weight="balanced"))])
params={"model__n_estimators":[100,200],"model__max_depth":[None,5,10],"model__min_samples_split":[2,5],"model__min_samples_leaf":[1,2]}
search=GridSearchCV(pipe,params,cv=cv,scoring="accuracy",n_jobs=-1)
search.fit(X_train,y_train); best=search.best_estimator_; pred=best.predict(X_test)
print("Best parameters:",search.best_params_); print("Best CV accuracy:",search.best_score_)
print("Test accuracy:",accuracy_score(y_test,pred))
print("Macro precision:",precision_score(y_test,pred,average="macro"))
print("Macro recall:",recall_score(y_test,pred,average="macro"))
print("Macro F1:",f1_score(y_test,pred,average="macro"))
print(classification_report(y_test,pred))
ConfusionMatrixDisplay.from_estimator(best,X_test,y_test,cmap="Blues"); plt.show()

## Conclusion, limitations and future work
This project demonstrates an end-to-end workflow on a public benchmark. Its small size and curated nature limit generalization. Future work could evaluate external wine samples, compare additional algorithms, calibrate probabilities, conduct post-hoc statistical tests, and investigate model errors.